# 🚀 Week 1 Exercise  

## 🧠 LLM-Based Technical Explainer Tool  
A lightweight AI-powered tool that takes technical questions and generates **concise, interview-ready explanations** using:

- ☁️ OpenAI (cloud-based model)  
- 💻 Ollama (locally hosted LLMs)  

It streams responses in real-time, making it ideal for **learning, interview prep, and quick concept clarification**.

## ⚡ How to Use  
👉 Run all cells to:

- Compare responses from:
  - **GPT-5 Nano (cloud)**
  - **Llama 3.2:1B (local via Ollama)**  
- Observe differences in:
  - Response quality  
  - Speed / latency  
  - Output style  

## 🎯 Purpose  
This project demonstrates:
- Integration with OpenAI APIs  
- Local LLM execution using Ollama  
- Streaming responses in a notebook environment  
- Practical usage for technical interview preparation  

## 🛠️ Tech Stack  
- Python 🐍  
- OpenAI SDK  
- Ollama

## 💡 Outcome  
A reusable tool that helps you quickly understand complex technical topics with **clear, structured explanations**. Powered by both cloud and local AI models.

In [79]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [80]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2:1b'

In [81]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

# Create OpenAI instance to be used for both openAI and ollama
openai = OpenAI()

API key found and looks good so far!


In [82]:
question = """
How would you optimize a slow performing angular application version above Angular 19 (Angular nineteen) standalone components and not modules. Give 3 ways. Keep the anser under 250 words.
"""

In [83]:
# Get gpt-4o-mini to answer, with streaming

system_prompt = """
You are a technical assistant to help interview questions for a Senior/Principal Angular Engineer. 
You have to be relevant and concise about your answer but keep the most important and 
tricky perspectives of the question being asked and responsd accordingly with small 
examples of before and after improvement code or strategies
"""

user_prompt = "Here is my question. Please keep it relevant and consize preferably under 250 words but pro level: "

def stream_answer(query):
    stream = openai.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + query}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [84]:
stream_answer(question)

Three pragmatic optimizations for Angular 19+ with standalone components:

1) Route-level lazy loading + custom preloading
- What changes: load standalone components on demand, plus a preloading strategy for non-critical routes.
- Before: eager imports in root module.
- After: use loadComponent in routes and a small preloading strategy.
Code (illustrative):
- Routes
  - { path: 'dashboard', loadComponent: () => import('./dashboard/dashboard.component').then(m => m.DashboardComponent), data: { preload: true } }
- Preloader
  - export class ComponentPreloadStrategy implements PreloadingStrategy {
      preload(route: Route, load: () => Observable<any>) {
        return (route.data?.preload) ? load() : of(null);
      }
    }
This yields smaller initial payload and faster first paint; non-critical routes load in background.

2) Change detection + rendering optimizations
- What changes: switch to OnPush, use trackBy, and async pipes for observables.
- Before: default change detection; simple ngFor without trackBy.
- After:
  - @Component({ standalone: true, changeDetection: ChangeDetectionStrategy.OnPush })
  - Template: <div *ngFor="let item of items; trackBy: trackById">{{ item.name }}</div>
  - Component: trackById(index: number, item: Item) { return item.id; }
This dramatically reduces CD cycles for large lists and keeps renders stable with immutable data patterns.

3) Scoped DI with route/component providers
- What changes: isolate heavy providers to the route/component to shrink the root injector and improve DI resolution time.
- Before: global/root providers for heavy services.
- After:
  - Routes: { path: 'reports', loadComponent: () => import('./reports/reports.component').then(m => m.ReportsComponent), providers: [importProvidersFrom([HttpClientModule])] }
  - Standalone component can also declare its own providers.
This minimizes injector lookups for non-critical features and reduces bundle size impact on initial load.

In [85]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [86]:
# Get Llama 3.2:1b to answer

system_prompt = """
You are a technical assistant to help interview questions for a Senior/Principal Angular Engineer. 
You have to be relevant and concise about your answer but keep the most important and 
tricky perspectives of the question being asked and responsd accordingly with small 
examples of before and after improvement code or strategies
"""
user_prompt = "Here is my question. Please keep it relevant and consize preferably under 250 words but pro level: "

def stream_answer(query):
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt + query}
          ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [87]:
stream_answer(question)

To optimize a slow-performing Angular application, especially when using standalone components versus modules, consider the following strategies:

1. **Component optimization**: Use the `async pipe` in place of reactive forms to improve component rendering speed. This can be particularly effective for complex or long-running computations within components. For example:

```typescript
import { Component } from '@angular/core';

@Component({
  selector: 'app-root',
  template: `
    <button (click)="asyncFunction()">Do something</button>
  `,
})
export class AppComponent {
  asyncFunction() {
    const result = await doLongRunningOperation();
    console.log(result);
  }
}
```

Implementing the `async pipe` can significantly improve component rendering speed.

2. **Avoid unnecessary re-renders**: Ensure that any complex computations or data filtering within components are only performed when necessary, as some cases may involve multiple re-renders. For instance:

```typescript
import { Component } from '@angular/core';

@Component({
  selector: 'app-example',
  template: `
    <p *ngIf="data">Data available for display</p>
  `,
})
export class ExampleComponent {
  data = true;
}
```

Instead of `console.log(data)`, use the `*ngIf` directive to only render the paragraph when `data` is `true`.

3. **Tree shaking (minification)**: Optimize production files by removing unused or redundant code, such as unused types, directives, or pipe names. This can be achieved through tools like UglifyJS (a transpiler) or Babel. Additionally, consider using a build configuration like Webpack's `outFile` option to enable tree shaking during the build process.

By implementing these strategies, you can improve the performance of your slow angular application by up to 40-50% and reduce the overhead of rendering.